# 04 — Evaluation

Reproduces the headline numbers from `docs/results.md`: clean-FID, SSIM, and the 3-arm downstream protocol.

In [ ]:
from pathlib import Path
from cv_diffusion.evaluation.fid import compute_fid
from cv_diffusion.evaluation.ssim import compute_set_ssim

real_dir = Path('../data/processed/xbd_tiles/flood')
fake_dir = Path('../data/synthetic/flood')

fid_clean = compute_fid(real_dir, fake_dir, mode='clean')
print('clean-FID:', fid_clean)

ssim = compute_set_ssim(real_dir, fake_dir, image_size=256)
print('SSIM stats:', ssim)

In [ ]:
# 3-arm downstream protocol (real-only / real+synth / synth-only).
from cv_diffusion.evaluation.downstream import DownstreamProtocolConfig, run_downstream_protocol

cfg = DownstreamProtocolConfig(
    real_train_root='../data/processed/xbd_classifier/train',
    real_val_root='../data/processed/xbd_classifier/val',
    real_test_root='../data/processed/xbd_classifier/test',
    synthetic_root='../data/synthetic',
    output_root='../outputs/downstream',
    backbone='resnet18',
    epochs=20,
    batch_size=32,
)
result = run_downstream_protocol(cfg)
print('Δ accuracy =', result.delta_accuracy)
print('Δ macro-F1 =', result.delta_macro_f1)

In [ ]:
# Plot the per-arm accuracies as a simple bar chart (the Figure-2 of the report).
import matplotlib.pyplot as plt

labels = ['real-only', 'real+synth', 'synth-only']
accs = [result.real_only.get('test_accuracy', 0.0),
        (result.real_plus_synth or {}).get('test_accuracy', 0.0),
        (result.synth_only or {}).get('test_accuracy', 0.0)]
plt.bar(labels, accs)
plt.ylabel('Top-1 accuracy')
plt.ylim(0, 1)
plt.title('Downstream classifier — 3-arm protocol')
for x, v in enumerate(accs):
    plt.text(x, v + 0.01, f'{v:.2%}', ha='center')
plt.show()